# 02 — Cleaning, Preprocessing, Feature Selection & Baseline Classification

Notebook ini menjelaskan pembersihan untuk kebutuhan pemodelan, pemilihan fitur, pipeline preprocessing, dan eksperimen baseline. Targetnya adalah **label kondisi yang tercatat**, bukan kesiapan operasional aktual.


## 1. Strategi cleaning dan feature selection

### Cleaning untuk pemodelan
- Rapikan spasi pada kolom teks dan ubah string kosong menjadi missing value.
- Baris tanpa label `Kondisi` tidak dapat dipakai sebagai contoh supervised learning, sehingga dikeluarkan dari subset pemodelan dan jumlahnya dilaporkan.
- Nilai kosong pada fitur tidak dihapus massal; fitur kategorikal diimputasi dengan modus di dalam pipeline. Jika fitur numerik tersedia, median digunakan.
- Pembersihan dan imputasi di-fit hanya pada data latih melalui `Pipeline`/`ColumnTransformer` agar statistik data uji tidak bocor ke proses training.
- Baris yang tampak identik setelah `plat_nomor` dihapus tidak otomatis dihapus: bisa jadi kendaraan berbeda memiliki atribut yang sama. Namun, kemiripan rekaman bisa membuat train/test tidak sepenuhnya independen; hal ini menjadi keterbatasan evaluasi.

### Pemilihan fitur
**Target:** `Kondisi`.

**Dikeluarkan dari fitur:**
- `Kondisi`: target, tidak boleh masuk ke `X`.
- `periode_data`: nilainya konstan (2025), sehingga tidak memberi daya pembeda pada model.
- `plat_nomor`: identifier aset yang tidak ada pada salinan sanitasi dan tidak layak digunakan sebagai prediktor default.

**Fitur kandidat:** `wilayah`, `kecamatan`, `kelurahan`, `jenis_kendaraan`, `klasifikasi`, `fungsi`, dan `penempatan`. Fitur ini dipilih karena menggambarkan konteks administratif dan karakteristik kendaraan yang tersedia. Akan tetapi, fitur administratif/penempatan dapat mencerminkan proses pencatatan, sehingga perlu diuji secara kritis dan tidak ditafsirkan sebagai penyebab kerusakan.

Feature selection di tahap ini berbasis relevansi konseptual dan pengecualian target/identifier/fitur konstan, bukan klaim bahwa setiap fitur sudah terbukti bermanfaat. Evaluasi lanjutan dapat membandingkan model dengan dan tanpa fitur administratif.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             balanced_accuracy_score, f1_score, recall_score,
                             precision_score, accuracy_score)

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
df_source = pd.read_csv(ROOT / "data" / "processed" / "kendaraan_2025_sanitized.csv")
before_rows = len(df_source)
before_missing = int(df_source.isna().sum().sum())

# Normalisasi ringan tanpa menghapus rekaman yang tampak sama.
df = df_source.copy()
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

missing_target = int(df["Kondisi"].isna().sum())
df = df.dropna(subset=["Kondisi"]).copy()
y = df["Kondisi"].astype(str)

# Target, fitur konstan, dan identifier tidak digunakan sebagai prediktor.
excluded = ["Kondisi", "periode_data", "plat_nomor"]
X = df.drop(columns=excluded, errors="ignore")

print("Ringkasan cleaning untuk pemodelan:")
display(pd.DataFrame([
    {"tahap": "Sebelum", "baris": before_rows, "kolom": df_source.shape[1], "sel_missing": before_missing},
    {"tahap": "Sesudah normalisasi teks dan eksklusi target kosong",
     "baris": len(df), "kolom": df.shape[1], "sel_missing": int(df.isna().sum().sum())}
]).set_index("tahap"))
print("Baris tanpa target yang dikeluarkan dari pemodelan:", missing_target)
print("Baris duplikat identik pada fitur yang tersedia (tidak dihapus otomatis):", int(X.assign(Kondisi=y).duplicated().sum()))
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Fitur kategorikal:", cat_cols)
print("Fitur numerik:", num_cols)

## 2. Preprocessing dalam pipeline

Pemisahan train/test dilakukan secara stratifikasi agar proporsi label relatif terjaga. Seluruh imputasi dan encoding dipasang di dalam pipeline dan hanya dipelajari dari data latih. `OneHotEncoder(handle_unknown="ignore")` menangani kategori baru yang mungkin muncul di data uji tanpa error. Fitur numerik—jika ada—diimputasi dengan median; model berbasis pohon dan regresi logistik dibandingkan dengan baseline mayoritas.

Catatan: karena atribut yang sama dapat muncul pada beberapa kendaraan setelah identifier dihapus, random split bisa menempatkan rekaman mirip pada train dan test. Skor harus dianggap eksploratif. Jika ada identifier kelompok yang aman atau unit pengamatan yang dapat diverifikasi, pertimbangkan validasi berkelompok pada penelitian lanjutan.


In [ ]:
feature_audit = pd.DataFrame({
    "fitur": X.columns,
    "tipe": [str(X[c].dtype) for c in X.columns],
    "missing": [int(X[c].isna().sum()) for c in X.columns],
    "kategori_unik": [int(X[c].nunique(dropna=True)) for c in X.columns]
})
display(feature_audit)
print("Target yang digunakan:", y.name, "| kelas:", sorted(y.unique().tolist()))
print("Jumlah data untuk pemodelan:", len(X))


In [ ]:
preprocessor = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_cols)
], remainder="drop")

models = {
    "Dummy most_frequent": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced",
        min_samples_leaf=2, n_jobs=-1
    )
}
results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = (pipe, pred)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        "recall_RUSAK": recall_score(y_test, pred, pos_label="RUSAK", zero_division=0),
        "precision_RUSAK": precision_score(y_test, pred, pos_label="RUSAK", zero_division=0)
    })
metrics = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(metrics.round(4))

In [ ]:
best_name = metrics.iloc[0]["model"]
best_pipe, best_pred = fitted[best_name]
print("Model dengan macro-F1 tertinggi pada satu holdout:", best_name)
print(classification_report(y_test, best_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, labels=["BAIK", "RUSAK"], values_format="d")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

## Catatan evaluasi

1. Hasil holdout tunggal memiliki ketidakpastian; tambahkan cross-validation pada tahap lanjutan.
2. Periksa kemungkinan kebocoran target melalui `penempatan`, `fungsi`, atau kategori administratif yang mungkin merefleksikan proses pencatatan.
3. `recall_RUSAK` penting karena false negative dapat membuat entri RUSAK terklasifikasi BAIK.
4. Model tidak mengestimasi kesiapan real-time, biaya perbaikan, atau alokasi optimal.
5. Jangan gunakan oversampling sebelum pemisahan data; bila diperlukan, terapkan hanya di pipeline data latih.